In [1]:
# 必要なライブラリをimport
import dataclasses
from tqdm import tqdm
import time
import io
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup
import re
import pandas as pd
import requests
from urllib.request import urlopen

### レースIDを取得するコード
- 全通りの組み合わせでrace_idを作成していますが、対象外のレースが大量に含まれてしまい無駄な処理が発生するため、ご注意ください。

In [2]:
# レース対象のIDを作成するためのコード

# 必要な年のリストを指定
years = [2023, 2024]

# 競馬場のリスト（01〜10）
racecourses = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']

# レースIDを格納するリスト
race_ids = []

# 各年についてレースIDを作成
for year in years:
    # 開催回数（01〜07）
    for kaisai in range(1, 8):
        kaisai_str = f"{kaisai:02d}"  # 2桁に変換

        # 開催日（01〜12）
        for day in range(1, 13):
            day_str = f"{day:02d}"  # 2桁に変換

            # レース番号（01〜12）
            for race_num in range(1, 13):
                race_num_str = f"{race_num:02d}"  # 2桁に変換

                # 競馬場ごとにレースIDを作成
                for racecourse in racecourses:
                    race_id = f"{year}{racecourse}{kaisai_str}{day_str}{race_num_str}"
                    race_ids.append(race_id)

# 作成したレースIDを表示（必要に応じてコメントアウト）
for race_id in race_ids:
    print(race_id)

ストリーミング出力は最後の 5000 行に切り捨てられました。
202401040705
202402040705
202403040705
202404040705
202405040705
202406040705
202407040705
202408040705
202409040705
202410040705
202401040706
202402040706
202403040706
202404040706
202405040706
202406040706
202407040706
202408040706
202409040706
202410040706
202401040707
202402040707
202403040707
202404040707
202405040707
202406040707
202407040707
202408040707
202409040707
202410040707
202401040708
202402040708
202403040708
202404040708
202405040708
202406040708
202407040708
202408040708
202409040708
202410040708
202401040709
202402040709
202403040709
202404040709
202405040709
202406040709
202407040709
202408040709
202409040709
202410040709
202401040710
202402040710
202403040710
202404040710
202405040710
202406040710
202407040710
202408040710
202409040710
202410040710
202401040711
202402040711
202403040711
202404040711
202405040711
202406040711
202407040711
202408040711
202409040711
202410040711
202401040712
202402040712
202403040712
202404040712
202405

## レース結果を取得

In [3]:
# レース結果を取得するためのコード
def fetch_race_results(race_ids):

    def get_html_content(url):
        response = requests.get(url)
        response.encoding = "EUC-JP"
        return response.text

    def parse_race_data(html, race_id):
        # HTMLから最初のテーブルを取得し、列名の半角スペースを除去
        df = pd.read_html(io.StringIO(html))[0]
        df.columns = df.columns.str.replace(' ', '')

        # HTMLからレース情報部分のテキストを取得
        soup = BeautifulSoup(html, "html.parser")
        race_info_text = soup.find(class_="racedata fc").find("span").contents[0]

         # レース情報をDataFrameに追加
        df["race_title"] = [soup.find(class_="racedata fc").find("h1").text] * len(df)
        df["race_type"] = [race_info_text[0]] * len(df)
        df["race_turn"] = [race_info_text[1]] * len(df)
        df["course_len"] = [re.findall(r'\d{4}', race_info_text)[0]] * len(df)
        df["weather"] = [re.findall(r'天候\s*:\s*([^\/]+)', race_info_text.replace("\xa0", ""))[0]] * len(df)
        df["ground_condition"] = [re.findall(r'良|稍重|重|不良', race_info_text)[0]] * len(df)
        df["year"] = [re.findall(r'(\d{4})', soup.find(class_="smalltxt").contents[0])[0]] * len(df)
        df["date"] = [re.findall(r'(\d{1,2}月\d{1,2}日)', soup.find(class_="smalltxt").contents[0])[0]] * len(df)
        df["location"] = [re.findall(r'\d+回(..)', soup.find(class_="smalltxt").contents[0])[0]] * len(df)

        # 馬ID、騎手IDをDataFrameに追加
        df["horse_id"] = [re.findall(r"\d+", a["href"])[0] for a in soup.find("table", summary="レース結果").find_all("a", href=re.compile("^/horse"))]
        df["jockey_id"] = [re.findall(r"\d+", a["href"])[0] for a in soup.find("table", summary="レース結果").find_all("a", href=re.compile("^/jockey"))]
        df["race_id"] = [race_id] * len(df)

        return df

    race_data_dict = {} # レースデータを格納する辞書

    for race_id in tqdm(race_ids):
        time.sleep(1) # スクレイピングのマナーとして1秒待機
        print(f'Fetching data for race_id: {race_id}')

        try:
            url = f"https://db.netkeiba.com/race/{race_id}"
            html_content = get_html_content(url) # HTMLコンテンツを取得
            race_df = parse_race_data(html_content, race_id) # HTMLを解析してDataFrameを作成
            race_data_dict[race_id] = race_df # 辞書にDataFrameを追加
        except (IndexError, AttributeError):
            continue # 辞書にDataFrameを追加
        except Exception as e:
            print(f'Error occurred: {e}') # その他のエラーが発生した場合はエラーメッセージを表示
            break

    combined_results_df = pd.concat(race_data_dict.values())
    return combined_results_df

In [4]:
# レース結果を取得
race_results = fetch_race_results(race_ids[:100])

# 日付列を結合して新しい列を作成し、不要な列を削除
race_results['event_date'] = pd.to_datetime(race_results['year'].astype(str) + '年' + race_results['date'], format='%Y年%m月%d日').dt.strftime('%Y-%m-%d')
race_results = race_results.drop(['year', 'date'], axis=1)

# 列名の修正と並び替えを同時に実行
new_order = [
    'race_id', 'event_date', 'location', 'race_title', 'race_type', 'race_turn',
    'course_len', 'weather', 'ground_condition', 'finish_position', 'frame_number',
    'horse_number', 'horse_id', 'horse_name', 'sex_age', 'carried_weight',
    'jockey_id', 'jockey', 'time', 'difference', 'odds', 'popularity',
    'horse_weight', 'trainer'
]

race_results.columns = [
    'finish_position', 'frame_number', 'horse_number', 'horse_name', 'sex_age', 'carried_weight',
    'jockey', 'time', 'difference', 'odds', 'popularity', 'horse_weight', 'trainer', 'race_title',
    'race_type', 'race_turn', 'course_len', 'weather', 'ground_condition', 'location', 'horse_id',
    'jockey_id', 'race_id', 'event_date'
]

# 列を並び替え
race_results = race_results[new_order]

# データの保存
race_results.to_csv('race_results.csv',index=None)

  0%|          | 0/100 [00:00<?, ?it/s]

Fetching data for race_id: 202301010101
Fetching data for race_id: 202302010101
Fetching data for race_id: 202303010101
Fetching data for race_id: 202304010101
Fetching data for race_id: 202305010101
Fetching data for race_id: 202306010101
Fetching data for race_id: 202307010101
Fetching data for race_id: 202308010101
Fetching data for race_id: 202309010101
Fetching data for race_id: 202310010101
Fetching data for race_id: 202301010102
Fetching data for race_id: 202302010102
Fetching data for race_id: 202303010102
Fetching data for race_id: 202304010102
Fetching data for race_id: 202305010102
Fetching data for race_id: 202306010102
Fetching data for race_id: 202307010102
Fetching data for race_id: 202308010102
Fetching data for race_id: 202309010102
Fetching data for race_id: 202310010102
Fetching data for race_id: 202301010103
Fetching data for race_id: 202302010103
Fetching data for race_id: 202303010103
Fetching data for race_id: 202304010103
Fetching data for race_id: 202305010103


## 競走馬の過去の成績を取得

In [5]:
def login_and_get_session(mail, password):
    session = requests.Session()
    login_data = {
        'login_id': mail,
        'pswd': password
    }
    login_url = "https://regist.netkeiba.com/account/?pid=login&action=auth"
    response = session.post(login_url, data=login_data)

    if response.url != login_url:
        print("ログイン成功")
    else:
        print("ログイン失敗")
        raise Exception("ログインに失敗しました")

    return session

def fetch_horse_results(horse_id_list, session):
    horse_results = {}
    for horse_id in tqdm(horse_id_list):
        time.sleep(1)
        try:
            url = f'https://db.netkeiba.com/horse/{horse_id}'
            response = session.get(url)
            if response.status_code == 200:
                df_list = pd.read_html(response.content, encoding='euc-jp')
                df = df_list[3]
                if df.columns[0] == '受賞歴':
                    df = df_list[4]
                df['horse_id'] = [horse_id] * len(df)
                horse_results[horse_id] = df
            else:
                print(f"Failed to retrieve data for horse_id: {horse_id}")
        except IndexError:
            continue
        except Exception as e:
            print(e)
            break

    if horse_results:
        horse_results_df = pd.concat([horse_results[key] for key in horse_results])
    else:
        horse_results_df = pd.DataFrame()

    return horse_results_df

In [6]:
# レースに参加した馬のidを取得
horse_id_list = race_results['horse_id'].unique()

# メールアドレスとパスワードを設定
mail = "311.1999.kenta@gmail.com"
password = "ano12pmo"

# ログインしてセッションを取得
session = login_and_get_session(mail, password)

# 過去成績データをスクレイピング
horse_results_df = fetch_horse_results(horse_id_list, session)

# 不要な列を削除
horse_results_df = horse_results_df.drop(['映 像', '厩舎 ｺﾒﾝﾄ', '備考'],axis=1)

# 列名の変更
horse_results_df.columns = [
    'date', 'venue', 'weather', 'race_number', 'race_name', 'num_horses', 'frame_number', 'horse_number',
    'odds', 'popularity', 'finish_position', 'jockey', 'carried_weight', 'distance', 'track_condition',
    'track_index', 'time', 'difference', 'time_index', 'passing_order', 'pace', 'last_3f', 'horse_weight',
    'winner_or_2nd', 'prize_money', 'horse_id'
]

# 日付の変換(yyyy-mm-dd)に
horse_results_df['date'] = pd.to_datetime(horse_results_df['date'], format='%Y/%m/%d')

# yyyy-mm-dd形式に変換
horse_results_df['date'] = horse_results_df['date'].dt.strftime('%Y-%m-%d')

horse_results_df.to_csv('horse_results.csv',index=None)

ログイン成功


  0%|          | 0/1271 [00:00<?, ?it/s]

KeyboardInterrupt: 

## スピード指数を取得

In [7]:
def fetch_index(original_race_id_list):
    # ID変換の対応を保持
    id_mapping = {int(str(id)[2:]): id for id in original_race_id_list}
    converted_race_id_list = list(id_mapping.keys())

    all_index_list = []

    for race_id in tqdm(converted_race_id_list):
        time.sleep(1)
        i = 0
        index_list = []

        url = 'https://jiro8.sakura.ne.jp/index2.php?code=' + str(race_id)
        url_html = requests.get(url)
        html = BeautifulSoup(url_html.content, 'html.parser')
        RaceTable01 = html.findAll('table', {'class':'c1'})[0]

        for RaceTable01_tr in RaceTable01.findAll('tr'):
            if i >= 1:
                if len(RaceTable01_tr.findAll('td')) > 7:
                    uma_ban = RaceTable01_tr.findAll('td')[1].get_text()
                    uma_ban = int(uma_ban)
                    zensou = RaceTable01_tr.findAll('td')[8]

                    speed_index = 0
                    rasing_index = 0
                    pace_index = 0
                    leading_index = 0

                    if len(zensou.findAll('span', {'class':'sn22'})) != 0:
                        speed_index = zensou.findAll('span', {'class':'sn22'})[0].get_text()
                        speed_index = float(speed_index)

                        rasing_index = zensou.findAll('span', {'class':'sn22'})[1].get_text()
                        rasing_index = float(rasing_index)

                        pace_index = zensou.findAll('span', {'class':'sn22'})[2].get_text()
                        pace_index = float(pace_index)

                        leading_index = zensou.findAll('span', {'class':'sn22'})[3].get_text()
                        leading_index = float(leading_index)

                    temp_list = [id_mapping[race_id], uma_ban, speed_index, rasing_index, pace_index, leading_index]
                    index_list.append(temp_list)
            i += 1

        df_index = pd.DataFrame(index_list)
        df_index = df_index.set_axis(['race_id', 'uma_ban', 'speed_index', 'rasing_index', 'pace_index', 'leading_index'], axis=1)
        all_index_list.append(df_index)

    all_race_df = pd.concat(all_index_list, ignore_index=True)
    return all_race_df

In [8]:
# 先ほど取得したレースIDでは存在しないrace_idもあるため、race_resultsにある存在するrace_idを持ってくる
unique_race_id_list = race_results['race_id'].unique()

# スピード指数の取得
speed_results = fetch_index(unique_race_id_list)

speed_results.to_csv('speed_results.csv',index=None)

  0%|          | 0/100 [00:00<?, ?it/s]

## 過去の払戻金を取得

In [9]:
def fetch_return(race_id_list):

    return_tables = {}
    for race_id in tqdm(race_id_list):
        time.sleep(1)
        try:
            url = "https://db.netkeiba.com/race/" + race_id

            f = urlopen(url)
            html = f.read()
            html = html.replace(b'<br />', b'br')
            dfs = pd.read_html(html)
            df = pd.concat([dfs[1], dfs[2]])

            df['race_id'] = [race_id] * len(df)
            return_tables[race_id] = df
        except IndexError:
            continue
        except AttributeError:  # 存在しないrace_idでAttributeErrorになるページもあるので追加
            continue
        except Exception as e:
            print(e)
            break
        except:
            break

    # pd.DataFrame型にして一つのデータにまとめる
    return_tables_df = pd.concat([return_tables[key] for key in return_tables])
    return return_tables_df

In [10]:
# 先ほど取得したレースIDでは存在しないrace_idもあるため、race_resultsにある存在するrace_idを持ってくる
unique_race_id_list = race_results['race_id'].unique()

# 払い戻し表データを取得
results = fetch_return(unique_race_id_list)

# 列名を指定された名前に変更
results.columns = ['baken_types', 'horse_number', 'refund', 'popularity', 'race_id']

# race_id列を最初の列に移動
results = results[['race_id', 'baken_types', 'horse_number', 'refund', 'popularity']]

results.to_csv('pay_results.csv',index=None)

  0%|          | 0/100 [00:00<?, ?it/s]